In [20]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
python.exe -m pip install --upgrade pip

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
import seaborn as sns

In [2]:
df=pd.read_csv("fake_job_postings.csv")

In [3]:
df.isnull().sum()

job_id                     0
title                      0
location                 346
department             11547
salary_range           15012
company_profile         3308
description                1
requirements            2696
benefits                7212
telecommuting              0
has_company_logo           0
has_questions              0
employment_type         3471
required_experience     7050
required_education      8105
industry                4903
function                6455
fraudulent                 0
dtype: int64

In [4]:
df["location"]=df["location"].fillna("unknown")
df["department"]=df["department"].fillna("unknown")
df["salary_range"]=df["salary_range"].fillna("Not specified")
df["employment_type"]=df["employment_type"].fillna("Not specified")
df["required_experience"]=df["required_experience"].fillna("Not specified")
df["required_education"]=df["required_education"].fillna("Not specified")
df["industry"]=df["industry"].fillna("Not specified")
df["function"]=df["function"].fillna("Not specified")

In [5]:
df.isnull().sum()

job_id                    0
title                     0
location                  0
department                0
salary_range              0
company_profile        3308
description               1
requirements           2696
benefits               7212
telecommuting             0
has_company_logo          0
has_questions             0
employment_type           0
required_experience       0
required_education        0
industry                  0
function                  0
fraudulent                0
dtype: int64

In [6]:

df['combined_text'] = df[['title', 'location', 'salary_range', 'company_profile', 'description', 
                          'requirements', 'benefits', 'employment_type', 
                          'required_experience', 'required_education', 'industry', 
                          'function', 'department']].apply(
    lambda x: ' '.join(x.fillna('').astype(str)), axis=1
)

In [7]:

text_columns = ['title', 'company_profile', 'description', 'requirements', 'benefits']
df[text_columns] = df[text_columns].fillna('Missing')

In [8]:
df.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,combined_text
0,1,Marketing Intern,"US, NY, New York",Marketing,Not specified,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,Missing,0,1,0,Other,Internship,Not specified,Not specified,Marketing,0,"Marketing Intern US, NY, New York Not specifie..."
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,Not specified,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,Not specified,Marketing and Advertising,Customer Service,0,"Customer Service - Cloud Video Production NZ, ..."
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",unknown,Not specified,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,Missing,0,1,0,Not specified,Not specified,Not specified,Not specified,Not specified,0,"Commissioning Machinery Assistant (CMA) US, IA..."
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,Not specified,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0,"Account Executive - Washington DC US, DC, Wash..."
4,5,Bill Review Manager,"US, FL, Fort Worth",unknown,Not specified,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0,"Bill Review Manager US, FL, Fort Worth Not spe..."


In [9]:
#we dont need the other columns now as all have been combined into one single text in a column


In [10]:

df.drop(columns=['title',
                 'location',
                 'salary_range',
                 'company_profile',
                 'description',
                 'requirements',
                 'benefits',
                 'employment_type',
                 'required_experience',
                 'required_education',
                 'industry',
                 'function',
                 'department',
                 'job_id',
                 'has_questions',
                 'telecommuting'], inplace=True)

In [11]:
print(df.columns)

Index(['has_company_logo', 'fraudulent', 'combined_text'], dtype='object')


In [12]:
def clean_text(text):
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = text.lower().strip()
    return text

In [13]:
df['combined_text']=df['combined_text'].apply(clean_text)

In [14]:
#LabelEncoder is meant for target labels y, not for normal input feature columns X. In your case, using it on fraudulent is correct because that column is the target

In [15]:
#le.fit_transform(df['fraudulent']) makes the encoder look at the values in df['fraudulent'], assign each unique class a number, and return the encoded result.

In [16]:
le=LabelEncoder()
df['fraudulent']=le.fit_transform(df['fraudulent'])

In [29]:
MAX_VOCAB_SIZE=10000
MAX_SEQUENCE_LENGTH=200
EMBEDDING_DIM=100

In [37]:
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
tokenizer.fit_on_texts(df['combined_text'])#Builds word-index mapping.
sequences=tokenizer.texts_to_sequences(df['combined_text'])#Converts text → integer sequences.
X=pad_sequences(sequences,maxlen=MAX_SEQUENCE_LENGTH)

In [38]:
X=pad_sequences(sequences,maxlen=MAX_SEQUENCE_LENGTH)
y=df['fraudulent']

In [39]:
smote=SMOTE(random_state=42)
X_resampled,y_resampled=smote.fit_resample(X,y)

In [40]:
print("original dataset shape", X.shape,y.shape)
print("after smote resampling shape", X_resampled.shape,y_resampled.shape)

original dataset shape (17880, 200) (17880,)
after smote resampling shape (34028, 200) (34028,)


In [46]:
X_train,X_test,y_train,y_test=train_test_split(X_resampled,y_resampled,test_size=0.2,random_state=42)

In [47]:
lstm_model = Sequential([
    Embedding(MAX_VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(32, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid')
])

In [49]:
lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_history = lstm_model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/5
851/851 ━━━━━━━━━━━━━━━━━━━━ 466s 533ms/step - accuracy: 0.9890 - loss: 0.0336 - val_accuracy: 0.9815 - val_loss: 0.0597
Epoch 2/5
851/851 ━━━━━━━━━━━━━━━━━━━━ 494s 580ms/step - accuracy: 0.9934 - loss: 0.0219 - val_accuracy: 0.9753 - val_loss: 0.0899
Epoch 3/5
851/851 ━━━━━━━━━━━━━━━━━━━━ 435s 511ms/step - accuracy: 0.9950 - loss: 0.0181 - val_accuracy: 0.9783 - val_loss: 0.1046
Epoch 4/5
851/851 ━━━━━━━━━━━━━━━━━━━━ 463s 535ms/step - accuracy: 0.9961 - loss: 0.0135 - val_accuracy: 0.9822 - val_loss: 0.0904
Epoch 5/5
851/851 ━━━━━━━━━━━━━━━━━━━━ 469s 551ms/step - accuracy: 0.9958 - loss: 0.0121 - val_accuracy: 0.9835 - val_loss: 0.0707


In [50]:
print("LSTM Model Evaluation")
lstm_predictions = (lstm_model.predict(X_test) > 0.5).astype(int)
print(classification_report(y_test, lstm_predictions))
print("LSTM F1 Score:", f1_score(y_test, lstm_predictions))

LSTM Model Evaluation
213/213 ━━━━━━━━━━━━━━━━━━━━ 36s 157ms/step
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      3396
           1       1.00      0.97      0.98      3410

    accuracy                           0.98      6806
   macro avg       0.98      0.98      0.98      6806
weighted avg       0.98      0.98      0.98      6806

LSTM F1 Score: 0.9833827893175074


In [53]:

lstm_model.save("Fake_job_detection.h5")

In [54]:

import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)